## Vaults for Secret Storage with Claude Managed Agents

### Installing Utilities and Libraries

In [1]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")
github_pat = os.getenv("GITHUB_PAT")

### Creating the Anthropic Client

In [3]:
import anthropic

client = anthropic.Anthropic(api_key=claude_api_key)

### Create the Vault

In [4]:
vault = client.beta.vaults.create(
    display_name="GitHub PAT Vault",
    metadata={
        "github_username": "ENTER-YOUR-GITHUB-USERNAME-HERE"
    }
)

print(vault.id)

vlt_011CfLFp46v88xeXq1WFi7uo


### Register the GitHub PAT Secret in the Vault

In [6]:
credential = client.beta.vaults.credentials.create(

    vault_id=vault.id,

    display_name="GitHub PAT",

    auth={
        "type": "static_bearer",

        "mcp_server_url": "https://api.githubcopilot.com/mcp/",

        "token": "github_pat"
    }
)

### Create the Agent

In [ ]:
agent = client.beta.agents.create(

    name="GitHub Assistant",

    model=claude_model_name,

    system="""
        You are an expert GitHub assistant.
    """,

    mcp_servers=[
        {
            "type": "url",
            "name": "GitHub",
            "url": "https://api.githubcopilot.com/mcp/"
        }
    ],

    tools=[
        {
            "type": "agent_toolset_20260401"
        },
        {
            "type": "mcp_toolset",
            "mcp_server_name": "GitHub",
            "default_config": {
                "permission_policy": {
                    "type": "always_allow"
                }
            }
        }
    ]
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### Create the Environment

In [ ]:
environment = client.beta.environments.create(
    name="quickstart-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

### Create a Session

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    vault_ids=[vault.id]
)

### Execute the Agent

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": """Tell me contents included in the GitHub Repo - kuljotSB/Claude-Certified-Developer-CCDV-F""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break